# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

**Note:** In `mlcroissant`, the record sets, fields, and columns are accessible via the Croissant schema. Each entity has an `@id` that uniquely identifies it. We'll list all available record sets and, for each, its fields and columns by `@id`.

In [ ]:
# List all record sets and their fields by @id
from collections import defaultdict

print("Available record sets (@id):\n")
record_sets = list(dataset.record_sets)
for record_set in record_sets:
    print(f"- RecordSet @id: {record_set.id} \n  Name: {getattr(record_set, 'name', '')}")
    print("  Fields:")
    for field in getattr(record_set, 'fields', []):
        print(f"    - Field @id: {field.id}    Name: {getattr(field, 'name', '')}    DataType: {getattr(field, 'data_type', '')}")
        if hasattr(field, 'columns') and field.columns:
            print("      Columns:")
            for col in field.columns:
                print(f"        - Column @id: {col.id}    Name: {getattr(col, 'name', '')}")
    print("")

# Store record set @ids for subsequent section
record_set_ids = [r.id for r in record_sets]
record_set_id_map = {r.id: r for r in record_sets}

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis. We'll use the record set and field `@id`s from the overview to ensure all references are fully explicit.

In [ ]:
# Extract data from each record set using its @id
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Reading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet '{record_set_id}'. Columns: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found for RecordSet '{record_set_id}'.\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records by value, normalizing numeric fields, and grouping by attribute. All data elements are referenced using their `@id`s as per the Croissant schema.

**Tips:** If you want to focus on a particular record set, select one from those listed above. We'll try working with the first available populated record set and analyze its numeric and categorical fields.

In [ ]:
# Choose the first record set with non-empty data for demo
record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        record_set_id = rid
        break
if record_set_id is None:
    print('No data available in any record set.')
else:
    df = dataframes[record_set_id]
    print(f"Using record set: {record_set_id}")

    # Identify numeric fields by @id (fields with dtype float or int)
    numeric_fields = []
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_fields.append(col)

    print(f"Numeric fields in RecordSet {record_set_id}: {numeric_fields}")
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Analyzing numeric field: {numeric_field_id}")

        threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
        print(filtered_df.head())

        # Avoid chained assignment warning with .copy()
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a non-numeric (categorical) field to group by
        group_field_id = None
        for col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]) and col != numeric_field_id:
                group_field_id = col
                break

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:\n")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print('No numeric fields found in the selected record set.')

## 5. Visualization
Visualize the distribution of a numeric field and its grouping by a categorical field (if available).

We'll use matplotlib and seaborn for plotting (install if required).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id} in RecordSet {record_set_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Grouped barplot if group_field_id exists
    if 'grouped_df' in locals() and group_field_id:
        plt.figure(figsize=(8, 5))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f'Mean of {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.show()

## 6. Conclusion
We have explored the FAIR^2 dataset using the `mlcroissant` library, listing record sets, fields, and extracting sample records by referencing their `@id`s as defined in the Croissant schema. We applied basic exploratory techniques, including value filtering and normalization, and visualized key data relationships.

Continue your analysis by examining other record sets, fields, and applying more domain-specific preprocessing and modeling as relevant to your research.